In [ ]:
from pathlib import Path
from PIL import Image
import json
import matplotlib.pyplot as plt
import json
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
import pandas as pd
from PIL import Image
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt


In [ ]:
# very first try to load and look at image data and annotations data 
image_path = Path("train/00000.jpg")
json_path = Path("train/00000.json")

# Load image
image = Image.open(image_path).convert("RGB")

print("Image size:", image.size)

# Load annotations
with open(json_path, "r") as f:
    annotations = json.load(f)

print("Annotation keys:")
for key in annotations.keys():
    print("-", key)

In [ ]:
# getting a sense of the dimensionality of the annotations in the dataset 
train_dir = Path("train")

annotation_counts = Counter()

json_files = list(train_dir.glob("*.json"))

for json_path in json_files:

    with open(json_path, "r") as f:
        annotations = json.load(f)

    for label in annotations.keys():

        # remove accidental spaces
        label = label.strip()

        annotation_counts[label] += 1

print(len(annotation_counts))

In [ ]:
# since this is just tabular data, easy to get a feel for it 
annotation_df = pd.DataFrame(
    annotation_counts.items(),
    columns=["annotation", "num_images"]
)

annotation_df = annotation_df.sort_values(
    "num_images",
    ascending=False
).reset_index(drop=True)

annotation_df["percentage"] = (
    annotation_df["num_images"]
    / len(json_files)
    * 100
)

annotation_df

In [ ]:
labels_per_image = []

for json_path in json_files:

    with open(json_path, "r") as f:
        annotations = json.load(f)

    labels_per_image.append(len(annotations))

print("Minimum:", np.min(labels_per_image))
print("Maximum:", np.max(labels_per_image))
print("Mean:", np.mean(labels_per_image))
print("Median:", np.median(labels_per_image))

In [ ]:
# lets see how one image with the notations on is 

image_path = Path("train/14655.jpg")
json_path = Path("train/14655.json")

image = Image.open(image_path).convert("RGB")
width, height = image.size

with open(json_path, "r") as f:
    annotations = json.load(f)

# Make figure aspect ratio match the actual image
fig_width = 12
fig_height = fig_width * height / width

fig, ax = plt.subplots(figsize=(fig_width, fig_height))

ax.imshow(image)

for label, points in annotations.items():

    xs = []
    ys = []

    for point in points:
        x_px = point["x"] * width
        y_px = point["y"] * height

        xs.append(x_px)
        ys.append(y_px)

    ax.plot(xs, ys, linewidth=2 )

    if len(xs) > 0:
        ax.text(
            xs[0],
            ys[0],
            label,
            fontsize=8
        )

ax.axis("off")

# Force axes to occupy the entire canvas
ax.set_position([0, 0, 1, 1])

# Force limits to exactly match the image
ax.set_xlim(0, width)
ax.set_ylim(height, 0)

plt.show()

In [ ]:
# manually inspecting another randomly picked image to see if
# the annotations are actually correctly placed and I didn't just 
# get lucky 

# lets see how one image with the notations on is 

image_path = Path("train/14634.jpg")
json_path = Path("train/14634.json")

image = Image.open(image_path).convert("RGB")
width, height = image.size

with open(json_path, "r") as f:
    annotations = json.load(f)

plt.figure(figsize=(12, 6.75))
plt.imshow(image)

for label, points in annotations.items():

    xs = []
    ys = []

    for point in points:
        x_px = point["x"] * width
        y_px = point["y"] * height

        xs.append(x_px)
        ys.append(y_px)

    # Draw line / curve
    plt.plot(xs, ys, linewidth=2)

    # Put label close to first point
    if len(xs) > 0:
        plt.text(
            xs[0],
            ys[0],
            label,
            fontsize=8
        )

plt.axis("off")
plt.show()

In [ ]:


image_path = "train/14655.jpg"
json_path = "train/14655.json"

# Load image
image = Image.open(image_path).convert("RGB")
width, height = image.size

# Load annotations
with open(json_path, "r") as f:
    annotations = json.load(f)

# Empty black mask
mask = np.zeros((height, width), dtype=np.uint8)

for label, points in annotations.items():

    pixel_points = []

    for point in points:

        x = int(point["x"] * width)
        y = int(point["y"] * height)

        pixel_points.append([x, y])

    pixel_points = np.array(pixel_points, dtype=np.int32)

    # Draw connected line if we have >= 2 points
    if len(pixel_points) >= 2:

        cv2.polylines(
            mask,
            [pixel_points],
            isClosed=False,
            color=255,
            thickness=4
        )

fig, ax = plt.subplots(figsize=(12, 6.75))

ax.imshow(mask, cmap="gray")
ax.axis("off")

plt.subplots_adjust(
    left=0,
    right=1,
    top=1,
    bottom=0
)

plt.show()

In [ ]:
#same as before check with a random observation: 


image_path = "train/00002.jpg"
json_path = "train/00002.json"

# Load image
image = Image.open(image_path).convert("RGB")
width, height = image.size

# Load annotations
with open(json_path, "r") as f:
    annotations = json.load(f)

# Empty black mask
mask = np.zeros((height, width), dtype=np.uint8)

for label, points in annotations.items():

    pixel_points = []

    for point in points:

        x = int(point["x"] * width)
        y = int(point["y"] * height)

        pixel_points.append([x, y])

    pixel_points = np.array(pixel_points, dtype=np.int32)

    # Draw connected line if we have >= 2 points
    if len(pixel_points) >= 2:

        cv2.polylines(
            mask,
            [pixel_points],
            isClosed=False,
            color=255,
            thickness=4
        )

fig, ax = plt.subplots(figsize=(12, 6.75))

ax.imshow(mask, cmap="gray")
ax.axis("off")

plt.subplots_adjust(
    left=0,
    right=1,
    top=1,
    bottom=0
)

plt.show()

# Object detection (player detection) baseline from pre-trained model

In [ ]:
pip install ultralytics 

In [ ]:
from ultralytics import YOLO 
model = YOLO('yolo11n.pt')

In [ ]:
results = model('train/00002.jpg')
img = results[0].plot()



plt.figure(figsize=(14, 8))
plt.imshow(img[..., ::-1])
plt.axis("off")
plt.show()

In [ ]:
result = results[0]
player_points = []
for box in result.boxes:

    class_id = int(box.cls[0])

    class_name = model.names[class_id]

    if class_name == "person":
        print(box.xyxy)
        
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    foot_x = (x1 + x2) / 2 # should approximate basically the position of the feet of the player (still an approximation of course it depends on what the player is doing and how accurate is the box around him)
    foot_y = y2 # ofc we don't change the y_level, still we assume they have their feet on the ground
    player_points.append((foot_x, foot_y))

player_points #for every player, this is an approximation of each player's position

In [ ]:
image = Image.open("train/00002.jpg").convert("RGB")

fig, ax = plt.subplots(figsize=(14, 8))

ax.imshow(image, cmap="gray")

for box in result.boxes:

    class_id = int(box.cls[0])

    if model.names[class_id] == "person":

        x1, y1, x2, y2 = box.xyxy[0].cpu().tolist()

        rect = Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            linewidth=2,
            edgecolor="red",
            facecolor="none"
        )

        ax.add_patch(rect)

        foot_x = (x1 + x2) / 2
        foot_y = y2

        ax.scatter(
            foot_x,
            foot_y,
            s=40,
            color="blue",
            zorder=3
        )

ax.set_xlim(0, width)
ax.set_ylim(height, 0)

ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
for box in result.boxes:
    class_id = int(box.cls[0])

    if model.names[class_id] == "person":
        x1, y1, x2, y2 = box.xyxy[0].cpu().tolist()
        conf = float(box.conf[0])

        foot_x = (x1 + x2) / 2
        foot_y = y2

        print(
            f"box=({x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f}) "
            f"conf={conf:.3f} "
            f"foot=({foot_x:.1f}, {foot_y:.1f})"
        )

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
ax.imshow(image)

person_id = 0

for box in result.boxes:

    class_id = int(box.cls[0])

    if model.names[class_id] == "person":

        x1, y1, x2, y2 = box.xyxy[0].cpu().tolist()
        conf = float(box.conf[0])

        foot_x = (x1 + x2) / 2
        foot_y = y2

        rect = Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            linewidth=2,
            edgecolor="red",
            facecolor="none"
        )

        ax.add_patch(rect)

        ax.scatter(
            foot_x,
            foot_y,
            s=40,
            color="blue"
        )

        ax.text(
            x1,
            y1,
            f"{person_id} ({conf:.2f})",
            fontsize=9,
            color="yellow"
        )

        person_id += 1

ax.set_xlim(0, width)
ax.set_ylim(height, 0)
ax.axis("off")

plt.show()

In [ ]:
import git 

In [ ]:
git status